#### CLEANING LOG


> - Loaded 50 rows, 4 columns (3 labeled columns)
> - Contained missing assault value, impossible UrbanPop value
> - Extreme UrbanPop and Assault values
> - Added a flag reasoning column

> Summary: Decided to flag all four, for a few reasons. Avoided deleting any rows mainly because these are info for all 50 states, so for someone reviewing/analyzing, seeing one of the states missing from the records could be quite alarming, as opposed to just a few sales information missing. The mean vs the median difference is about 23.18, and it's mainly because of the outlier - so referring to the median should be a good enough comprehension for where the assault values would lie.
> 
> So a flag column with the reasoning has been provided for the flagged rows, labelled "flag-reason". Doing this method instead of filling in missing values or removing them prevents me from messing with state data especially when neigboring stats are reasonable and don't make the entire row seem invalid and leaves a clear view for the analyst going through this notebook. Corrections can be made easily, and extreme values can be verified and researched to confirm.

In [111]:
import pandas as pd

In [112]:
df = pd.read_csv("usarrests.csv")

print(df.head(10), "\n")
print(df.describe(), "\n")
print(df.dtypes, "\n")
print(df.isna().sum(), "\n")
print(df.shape)

    Unnamed: 0  Murder  Assault  UrbanPop
0      Alabama    13.2    236.0        58
1       Alaska    10.0    263.0        48
2      Arizona     8.1    294.0        80
3     Arkansas     8.8    190.0        50
4   California     9.0    276.0        91
5     Colorado     7.9    204.0        78
6  Connecticut     3.3    110.0        77
7     Delaware     5.9    238.0        72
8      Florida    15.4    335.0        80
9      Georgia    17.4      NaN        60 

         Murder     Assault   UrbanPop
count  50.00000   49.000000   50.00000
mean    7.78800  182.183673   74.20000
std     4.35551  130.877435   73.40828
min     0.80000   45.000000    6.00000
25%     4.07500  109.000000   53.25000
50%     7.25000  159.000000   66.00000
75%    11.25000  249.000000   77.75000
max    17.40000  879.000000  570.00000 

Unnamed: 0        str
Murder        float64
Assault       float64
UrbanPop        int64
dtype: object 

Unnamed: 0    0
Murder        0
Assault       1
UrbanPop      0
dtype: int64 



> Judging from this data, first thing to notice is urban population %, it's above 100. there is one with 570, which is impossible.
It would suggest the statistic relating to more than the entire population of that country itself, which doesn't make sense.
>
> Assault seems to have one giant outlier, sitting at 879, seems like an extreme value considering the mean is around 182 (probably 
got dragged up by the outlier) meaning a more accurate mean would be lower. 879 blows way past the upper quartiles too, which leads
me to believe it's an extreme value. As for one more impossible value, there is an empty row for Assault, which wouldn't be very
believable due to its surrouding statistics given UrbanPop as 60% and 17.4 for Murder.
>
> Another extreme value could be the minimum for UrbanPop, it's at 6%. Unsure for which state, so we can check.

In [113]:
print(f"Low Urban Populations (<20%):\n{df[df["UrbanPop"] < 20]}\n")
print(f"Impossible Urban Populations (>100%):\n{df[df["UrbanPop"] > 100]}\n")

Low Urban Populations (<20%):
   Unnamed: 0  Murder  Assault  UrbanPop
31   New York    11.1    254.0         6

Impossible Urban Populations (>100%):
   Unnamed: 0  Murder  Assault  UrbanPop
14       Iowa     2.2     56.0       570




> Given that New York has one of the highest urban populations, a 6% seems logically impossible (but mathematically possible, so we'll label it extreme). We can also 
see that Iowa is the one with a explicitly impossible urban population, even when their counts of murder and assault are relatively low compared to New York who 
supposedly has a 6% and 254 counts of assault and over 5x the counts of murder.
> 
> Going to run a couple more checks:


In [114]:
print(f"High Assault rates (above Upper Quartile >75%):\n{df[df["Assault"] > 254]}")

High Assault rates (above Upper Quartile >75%):
        Unnamed: 0  Murder  Assault  UrbanPop
1           Alaska    10.0    263.0        48
2          Arizona     8.1    294.0        80
4       California     9.0    276.0        91
8          Florida    15.4    335.0        80
19        Maryland    11.3    300.0        67
21        Michigan    12.1    255.0        74
23     Mississippi    16.1    259.0        44
30      New Mexico    11.4    285.0        70
32  North Carolina    13.0    337.0        45
39  South Carolina    14.4    879.0        48


> Given these checks, we can store these flagged rows in a variable.

In [115]:
# Used Gemini for this syntax using bitwise operator, wasn't sure it was possible (Asked how to get a list of specific rows in one variable)
flagged = df[(df["UrbanPop"] < 20) | (df["UrbanPop"] > 100) | (df["Assault"] > 400) | df["Assault"].isna()]
print(flagged)

        Unnamed: 0  Murder  Assault  UrbanPop
9          Georgia    17.4      NaN        60
14            Iowa     2.2     56.0       570
31        New York    11.1    254.0         6
39  South Carolina    14.4    879.0        48



> We'll flag the the extreme values (Assault 879 and UrbanPop 6), as they could have been intentional and would probably be better if they 
go through review. Their values aren't impossible, but they are extremely questionable given surrounding statistics, such as population 
and murder.


In [ ]:
df_clean = df.copy() # Didn't want a destructive structure and add every change to the original df, so asked Gemini for a non-destructive method and was shown the .copy() method

# Also got Gemini's help for this structure, didn't know I can make a flag column like this
# Apparently it's safer and better than double brackets (also less complicated, .loc selects rows based on
# condition, and then which column to change for those rows)
df_clean["flag-reason"] = "OK"
df_clean.loc[df_clean["UrbanPop"] > 100, "flag-reason"] = "Impossible UrbanPop (>100%)"
df_clean.loc[df_clean["UrbanPop"] < 20, "flag-reason"] = "Extreme low UrbanPop"
df_clean.loc[df_clean["Assault"] > 400, "flag-reason"] = "Extreme high Assault"
df_clean.loc[df_clean["Assault"].isna(), "flag-reason"] = "Missing Assault (NaN)"

print(df_clean[df_clean["flag-reason"] != "OK"])

        Unnamed: 0  Murder  Assault  UrbanPop                  flag-reason
9          Georgia    17.4      NaN        60        Missing Assault (NaN)
14            Iowa     2.2     56.0       570  Impossible UrbanPop (>100%)
31        New York    11.1    254.0         6         Extreme low UrbanPop
39  South Carolina    14.4    879.0        48         Extreme high Assault


In [ ]:
df_clean.to_csv("cleaned_usarrests.csv", index=False)